# Comparaison d'algos en Classification Supervisée

Nous allons comparer les différentes méthodes de classification supervisée n que nous avons vues jusqu'à présent, la régression logistique avec toutes les variables, puis  avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC puis les méthodes de vraisemblance pénalisée.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [55]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as sklm
from patsy import dmatrix

#import logistic_step_sk as lss 

In [56]:
don = pd.read_csv("dfbase.csv",header=0,sep=",")
#don = pd.read_csv("spambase.data",sep=",")
don.shape

(462, 10)

Codage des variables qualitatives famhist

In [57]:
X = don.drop(columns=["Y"])
#X = don.iloc[:,:56]
X = X.to_numpy()
Y = don["Y"].to_numpy()
#Y = don.iloc[:,57].to_numpy()

In [67]:
nb=10
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=123)
#PROB = pd.DataFrame({"Y":Y,"log":0.0,"BIC":0.0,"AIC":0.0,
#                    "ridge":0.0,"lasso":0.0,"elast":0.0,"arbre":0.0,"foret":0.0})
PROB = pd.DataFrame({"Y":Y,"log":0.0,"ridge":0.0,"lasso":0.0,"elast":0.0,"arbre":0.0,"foret":0.0,
                    "gbm1":0.0,"gbm2":0.0,"cat":0.0})

choix des grilles de régularisation

In [68]:
def grille(X, y, type = "lasso", ng=100):
    scalerX = StandardScaler().fit(X)
    Xcr= scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y-y.mean()))).max()/X.shape[0]
    llc = np.linspace(0,-4,ng)
    ll = l0*10**llc
    if type=="lasso":
        Cs = 1/ 0.9/ X.shape[0] / (l0*10**(llc))
    elif type=="ridge":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 100)
    elif type=="enet":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 2)
    return Cs

In [69]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

On compare les méthodes en utilisant skf

In [70]:
for app_index, val_index in skf.split(X,Y):
    Xapp = X[app_index,:]
    Xtest = X[val_index,:]
    Yapp = Y[app_index]
    ### logistique
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    #choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="bic",multi_class='auto').fit(Xapp,Yapp)
    #PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    #choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="aic",multi_class='auto').fit(Xapp,Yapp)
    #PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2",Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]
    ###arbre
    arbre = DecisionTreeClassifier(min_samples_leaf=5).fit(Xapp,Yapp)
    PROB.loc[val_index,"arbre"] = arbre.predict_proba(Xtest)[:,1]
    ###foret
    foret = RandomForestClassifier().fit(Xapp,Yapp)
    PROB.loc[val_index,"foret"] = foret.predict_proba(Xtest)[:,1]
    ###gradient boosting
    params = dict(n_estimators=1000, max_depth=1, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbm1"] = gbm_early_stopping.predict_proba(Xtest)[:,1]
    params = dict(n_estimators=1000, max_depth=2, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingClassifier(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PROB.loc[val_index,"gbm2"] = gbm_early_stopping.predict_proba(Xtest)[:,1]
    ### cat
    X_train, X_valid, y_train, y_valid = train_test_split(Xapp,Yapp,test_size=0.1,random_state=42)
    model = CatBoostClassifier(iterations=5000,learning_rate=0.01,subsample=0.1,loss_function="Logloss",
    eval_metric="Accuracy",random_seed=42,)
    model.fit(X_train,y_train,eval_set=(X_valid, y_valid),early_stopping_rounds=10)
    print(model.get_best_iteration())
    PROB.loc[val_index,"cat"] = model.predict_proba(Xtest)[:,1]

/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 

/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, 

63
24
0:	learn: 0.7506702	test: 0.7142857	best: 0.7142857 (0)	total: 1.75ms	remaining: 8.74s
1:	learn: 0.7801609	test: 0.7380952	best: 0.7380952 (1)	total: 2.8ms	remaining: 7s
2:	learn: 0.7667560	test: 0.7380952	best: 0.7380952 (1)	total: 3.81ms	remaining: 6.35s
3:	learn: 0.7774799	test: 0.7619048	best: 0.7619048 (3)	total: 4.85ms	remaining: 6.06s


4:	learn: 0.7801609	test: 0.7619048	best: 0.7619048 (3)	total: 5.84ms	remaining: 5.84s
5:	learn: 0.7694370	test: 0.7380952	best: 0.7619048 (3)	total: 7.51ms	remaining: 6.25s
6:	learn: 0.7694370	test: 0.6904762	best: 0.7619048 (3)	total: 8.56ms	remaining: 6.1s
7:	learn: 0.7855228	test: 0.7142857	best: 0.7619048 (3)	total: 9.77ms	remaining: 6.09s
8:	learn: 0.7882038	test: 0.7142857	best: 0.7619048 (3)	total: 10.9ms	remaining: 6.03s
9:	learn: 0.7855228	test: 0.7142857	best: 0.7619048 (3)	total: 12.1ms	remaining: 6.02s
10:	learn: 0.7908847	test: 0.7142857	best: 0.7619048 (3)	total: 13.1ms	remaining: 5.96s
11:	learn: 0.7935657	test: 0.6904762	best: 0.7619048 (3)	total: 14.2ms	remaining: 5.88s
12:	learn: 0.7935657	test: 0.6904762	best: 0.7619048 (3)	total: 15.1ms	remaining: 5.79s
13:	learn: 0.7989276	test: 0.6904762	best: 0.7619048 (3)	total: 16.1ms	remaining: 5.75s
Stopped by overfitting detector  (10 iterations wait)

bestTest = 0.7619047619
bestIteration = 3

Shrink model to first 4 itera

...

/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, 

38
12
0:	learn: 0.7352941	test: 0.6190476	best: 0.6190476 (0)	total: 3.88ms	remaining: 19.4s
1:	learn: 0.7486631	test: 0.6904762	best: 0.6904762 (1)	total: 7.57ms	remaining: 18.9s
2:	learn: 0.7860963	test: 0.6428571	best: 0.6904762 (1)	total: 10.6ms	remaining: 17.7s
3:	learn: 0.7807487	test: 0.6666667	best: 0.6904762 (1)	total: 15.3ms	remaining: 19.2s
4:	learn: 0.7780749	test: 0.6666667	best: 0.6904762 (1)	total: 20.5ms	remaining: 20.5s
5:	learn: 0.7807487	test: 0.6428571	best: 0.6904762 (1)	total: 24.8ms	remaining: 20.6s


6:	learn: 0.7780749	test: 0.6190476	best: 0.6904762 (1)	total: 77.5ms	remaining: 55.2s
7:	learn: 0.7834225	test: 0.6428571	best: 0.6904762 (1)	total: 93.8ms	remaining: 58.5s
8:	learn: 0.7914439	test: 0.6190476	best: 0.6904762 (1)	total: 95.7ms	remaining: 53.1s
9:	learn: 0.7780749	test: 0.6190476	best: 0.6904762 (1)	total: 140ms	remaining: 1m 9s
10:	learn: 0.7834225	test: 0.6428571	best: 0.6904762 (1)	total: 142ms	remaining: 1m 4s
11:	learn: 0.7780749	test: 0.6428571	best: 0.6904762 (1)	total: 145ms	remaining: 1m
Stopped by overfitting detector  (10 iterations wait)

bestTest = 0.6904761905
bestIteration = 1

Shrink model to first 2 iterations.


1


In [71]:
round(PROB.iloc[0:4,:],3)

,Y,log,ridge,lasso,elast,arbre,foret,gbm1,gbm2,cat
0,1,0.742,0.677,0.598,0.605,1.000,0.87,0.624,0.769,0.509
1,1,0.292,0.317,0.373,0.358,0.000,0.37,0.294,0.359,0.494
2,0,0.251,0.286,0.314,0.297,0.667,0.21,0.201,0.166,0.486
3,1,0.719,0.679,0.696,0.682,1.000,0.75,0.657,0.753,0.501


In [73]:
PROB.to_csv("PROB.csv",index=False)

In [29]:
PROB.to_csv("PROBpoly.csv",index=False)

In [36]:
PROB.to_csv("PROBinter.csv",index=False)